# NLP Modules Process Demonstration

This notebook demonstrates how the EduCompose NLP modules work, showing the step-by-step process of analyzing essays.

## Overview

The system uses 5 main NLP modules:
1. **Grammar Analyzer** - Detects grammatical errors and analyzes syntax
2. **Readability Analyzer** - Calculates readability metrics (Flesch, SMOG, etc.)
3. **Coherence Analyzer** - Analyzes text coherence using entity-grid and semantic similarity
4. **Argument Miner** - Extracts argument components using Toulmin's model
5. **Knowledge Graph Builder** - Builds semantic networks from concepts

Let's see how each works!


In [ ]:
# Install required packages
!pip install -q spacy language-tool-python textstat sentence-transformers networkx nltk

# Download spaCy model
!python -m spacy download en_core_web_lg -q

# Download NLTK data
import nltk
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

print("✅ Setup complete!")


In [ ]:
# Sample essay text for demonstration
sample_essay = """
Decarbonization and the Geopolitical Singularity: The Complexities of a Post-Fossil Fuel System

The discourse surrounding renewable energy has transcended simple advocacy; it is now centered on the techno-economic viability and geopolitical implications of achieving a post-fossil fuel energy system. The transition presents a challenge of systemic inertia, where the incumbent energy infrastructure acts as a massive sunk cost, complicating the accelerated adoption of intermittent, distributed generation sources.

A key analytical measure in this shift is the Levelized Cost of Energy (LCOE), which has seen solar photovoltaic (PV) and onshore wind fall below the cost of new-build fossil fuel plants in many jurisdictions. However, LCOE alone is an insufficient metric; System-Levelized Cost of Energy (SLCOE) and Value-Adjusted LCOE (VALCOE) provide a more accurate picture by integrating the costs associated with transmission upgrades, necessary storage capacity, and the value of dispatchability. The integration costs of VRE—specifically managing frequency response and voltage stability in grids with reduced rotational inertia—represent a highly technical challenge requiring sophisticated power electronics and grid-forming inverters.

Furthermore, the transition shifts resource dependence, trading hydrocarbon leverage for mineral resource dependency. The supply chains for critical rare earth elements (REEs), copper, cobalt, and lithium essential for renewable technologies introduce new geopolitical risks. However, these challenges are not insurmountable. Strategic investments in recycling, alternative materials, and diversified supply chains can mitigate these risks while advancing the transition toward sustainable energy systems.
"""

print(f"Sample essay ({len(sample_essay.split())} words):")
print("=" * 80)
print(sample_essay)


## 1. Grammar Analyzer Process

The Grammar Analyzer uses three main techniques:
1. **LanguageTool** - Rule-based grammar checking
2. **spaCy** - Syntactic analysis (dependency parsing, POS tagging)
3. **Basic Rules** - Custom pattern matching for common errors

### Step-by-Step Process:


In [ ]:
import spacy
from language_tool_python import LanguageTool
import re

# Load spaCy model
print("Step 1: Loading spaCy model...")
nlp = spacy.load("en_core_web_lg")
print("✅ spaCy loaded")

# Initialize LanguageTool
print("\nStep 2: Initializing LanguageTool...")
language_tool = LanguageTool('en-US')
print("✅ LanguageTool initialized")

# Process text
print("\nStep 3: Processing text with spaCy...")
doc = nlp(sample_essay)
sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]
print(f"✅ Found {len(sentences)} sentences")

# Step 4: Check grammar with LanguageTool
print("\nStep 4: Checking grammar with LanguageTool...")
grammar_errors = language_tool.check(sample_essay)
print(f"✅ Found {len(grammar_errors)} potential grammar issues")

# Display errors
print("\n" + "="*80)
print("GRAMMAR ERRORS DETECTED:")
print("="*80)
for i, error in enumerate(grammar_errors[:5], 1):  # Show first 5
    print(f"\nError {i}:")
    print(f"  Message: {error.message}")
    print(f"  Category: {error.category}")
    print(f"  Context: {error.context}")
    if error.replacements:
        print(f"  Suggestions: {error.replacements[:3]}")


In [ ]:
# Step 5: Analyze syntax patterns
print("\nStep 5: Analyzing syntax patterns with spaCy...")
print("="*80)

sentence_types = {"simple": 0, "compound": 0, "complex": 0, "compound_complex": 0}

for sent in doc.sents:
    num_verbs = len([token for token in sent if token.pos_ == "VERB"])
    num_conjunctions = len([token for token in sent if token.dep_ == "cc"])
    
    if num_verbs == 1 and num_conjunctions == 0:
        sentence_types["simple"] += 1
    elif num_verbs > 1 and num_conjunctions > 0:
        sentence_types["compound_complex"] += 1
    elif num_verbs > 1:
        sentence_types["complex"] += 1
    elif num_conjunctions > 0:
        sentence_types["compound"] += 1

print("Sentence Type Distribution:")
for stype, count in sentence_types.items():
    print(f"  {stype}: {count}")

# Calculate complexity score
total_sentences = len(list(doc.sents))
if total_sentences > 0:
    complexity_score = (
        sentence_types["complex"] * 2 +
        sentence_types["compound_complex"] * 3 +
        sentence_types["compound"] * 1
    ) / total_sentences * 100
    print(f"\nSyntax Complexity Score: {complexity_score:.2f}")

# Step 6: Calculate grammar score
error_density = len(grammar_errors) / len(sample_essay.split())
grammar_score = max(0.0, 100.0 - (error_density * 1000))
print(f"\nFinal Grammar Score: {grammar_score:.2f}/100")


## 2. Readability Analyzer Process

The Readability Analyzer calculates multiple metrics:
1. **Flesch Reading Ease** - Ease of reading (0-100, higher = easier)
2. **Flesch-Kincaid Grade Level** - U.S. grade level
3. **SMOG Index** - Reading level estimate
4. **Lexical Diversity** - Type-Token Ratio (vocabulary variety)

### Step-by-Step Process:


In [ ]:
import textstat

print("Step 1: Calculating readability metrics...")
print("="*80)

# Calculate Flesch Reading Ease
flesch_ease = textstat.flesch_reading_ease(sample_essay)
print(f"Flesch Reading Ease: {flesch_ease:.2f}")
print(f"  Interpretation: ", end="")
if flesch_ease >= 90:
    print("Very Easy (5th grade)")
elif flesch_ease >= 80:
    print("Easy (6th grade)")
elif flesch_ease >= 70:
    print("Fairly Easy (7th grade)")
elif flesch_ease >= 60:
    print("Standard (8th-9th grade)")
elif flesch_ease >= 50:
    print("Fairly Difficult (10th-12th grade)")
elif flesch_ease >= 30:
    print("Difficult (College)")
else:
    print("Very Difficult (College Graduate)")

# Calculate Grade Level
flesch_grade = textstat.flesch_kincaid_grade(sample_essay)
print(f"\nFlesch-Kincaid Grade Level: {flesch_grade:.2f}")

# Calculate SMOG Index
smog_index = textstat.smog_index(sample_essay)
print(f"SMOG Index: {smog_index:.2f}")

# Calculate Coleman-Liau Index
coleman_liau = textstat.coleman_liau_index(sample_essay)
print(f"Coleman-Liau Index: {coleman_liau:.2f}")


In [ ]:
# Step 2: Calculate lexical diversity
print("\nStep 2: Calculating lexical diversity...")
print("="*80)

words = sample_essay.split()
unique_words = set(word.lower() for word in words)
lexical_diversity = len(unique_words) / len(words) if words else 0

print(f"Total words: {len(words)}")
print(f"Unique words: {len(unique_words)}")
print(f"Lexical Diversity (Type-Token Ratio): {lexical_diversity:.2%}")

# Step 3: Calculate vocabulary sophistication
long_words = [w for w in words if len(w) > 6]
vocab_sophistication = len(long_words) / len(words) if words else 0
print(f"\nVocabulary Sophistication: {vocab_sophistication:.2%} (words > 6 chars)")

# Step 4: Calculate average sentence length
sentences = re.split(r'[.!?]+', sample_essay)
sentences = [s.strip() for s in sentences if s.strip()]
avg_sentence_length = len(words) / len(sentences) if sentences else 0
print(f"Average Sentence Length: {avg_sentence_length:.1f} words")

# Step 5: Calculate normalized readability score
if 50 <= flesch_ease <= 70:
    flesch_score = 100
elif 40 <= flesch_ease < 50 or 70 < flesch_ease <= 80:
    flesch_score = 80
elif 30 <= flesch_ease < 40 or 80 < flesch_ease <= 90:
    flesch_score = 60
else:
    flesch_score = 40

lexical_score = min(100, lexical_diversity * 200)

if 15 <= avg_sentence_length <= 20:
    sent_score = 100
elif 10 <= avg_sentence_length < 15 or 20 < avg_sentence_length <= 25:
    sent_score = 80
else:
    sent_score = 60

readability_score = (flesch_score * 0.5 + lexical_score * 0.3 + sent_score * 0.2)
print(f"\nFinal Readability Score: {readability_score:.2f}/100")


## 3. Coherence Analyzer Process

The Coherence Analyzer uses multiple techniques:
1. **Entity-Grid Model** - Tracks how entities (nouns) are mentioned across sentences
2. **Semantic Similarity** - Uses SentenceTransformer to measure sentence similarity
3. **Transition Analysis** - Identifies transitional words/phrases
4. **Paragraph Unity** - Measures how well sentences in a paragraph relate to topic sentence

### Step-by-Step Process:


In [ ]:
from collections import defaultdict
import numpy as np

print("Step 1: Entity-Grid Analysis")
print("="*80)

# Extract entities (nouns) from each sentence
entity_grid = defaultdict(lambda: defaultdict(int))
entity_positions = defaultdict(list)

sentences_list = [sent.text.strip() for sent in doc.sents if sent.text.strip()]

for i, sentence in enumerate(sentences_list):
    sent_doc = nlp(sentence)
    entities = []
    
    # Extract nouns and proper nouns
    for token in sent_doc:
        if token.pos_ in ["NOUN", "PROPN"] and not token.is_stop:
            entity = token.lemma_.lower()
            entities.append(entity)
            entity_positions[entity].append(i)
    
    print(f"\nSentence {i+1}: {sentence[:60]}...")
    print(f"  Entities found: {list(set(entities))[:5]}")

# Calculate entity continuation score
continuation_count = 0
total_entities = len(entity_positions)

for entity, positions in entity_positions.items():
    if len(positions) > 1:
        for j in range(len(positions) - 1):
            if positions[j+1] == positions[j] + 1:
                continuation_count += 1

if total_entities > 0:
    entity_grid_score = min(100, (continuation_count / total_entities) * 100)
else:
    entity_grid_score = 50.0

print(f"\nEntity Grid Score: {entity_grid_score:.2f}/100")
print(f"  Entities continuing across sentences: {continuation_count}")
print(f"  Total unique entities: {total_entities}")


In [ ]:
# Step 2: Semantic Similarity Analysis
print("\nStep 2: Semantic Similarity Analysis")
print("="*80)

from sentence_transformers import SentenceTransformer

print("Loading SentenceTransformer model...")
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded")

# Get sentence embeddings
print(f"\nEncoding {len(sentences_list)} sentences...")
embeddings = sentence_model.encode(sentences_list)

# Calculate cosine similarity between consecutive sentences
similarities = []
for i in range(len(embeddings) - 1):
    similarity = np.dot(embeddings[i], embeddings[i+1]) / (
        np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[i+1])
    )
    similarities.append(float(similarity))
    print(f"  Sentence {i+1} → {i+2}: {similarity:.3f}")

if similarities:
    avg_similarity = np.mean(similarities)
    semantic_score = avg_similarity * 100
    print(f"\nAverage Semantic Similarity: {avg_similarity:.3f}")
    print(f"Semantic Similarity Score: {semantic_score:.2f}/100")


In [ ]:
# Step 3: Transition Analysis
print("\nStep 3: Transition Analysis")
print("="*80)

transitions = {
    "addition": ["furthermore", "moreover", "additionally", "also", "and", "in addition"],
    "contrast": ["however", "nevertheless", "on the other hand", "in contrast", "but", "although"],
    "cause_effect": ["therefore", "thus", "consequently", "as a result", "because", "since"],
    "example": ["for example", "for instance", "specifically", "such as"],
    "conclusion": ["in conclusion", "to summarize", "in summary", "overall", "finally"]
}

found_transitions = []
for i, sentence in enumerate(sentences_list):
    sentence_lower = sentence.lower()
    for category, words in transitions.items():
        for word in words:
            if word in sentence_lower:
                found_transitions.append({
                    "sentence_index": i,
                    "category": category,
                    "word": word
                })
                print(f"  Sentence {i+1}: Found '{word}' ({category})")
                break

transition_score = min(100, (len(found_transitions) / len(sentences_list)) * 100) if sentences_list else 50
print(f"\nTransition Score: {transition_score:.2f}/100")
print(f"  Found {len(found_transitions)} transitions in {len(sentences_list)} sentences")


In [ ]:
# Step 4: Calculate Overall Coherence Score
print("\nStep 4: Overall Coherence Score Calculation")
print("="*80)

weights = {
    "entity_grid": 0.3,
    "semantic_similarity": 0.3,
    "transitions": 0.2,
    "paragraph_unity": 0.2  # Simplified for demo
}

# For demo, assume paragraph unity = 80
paragraph_unity = 80.0

coherence_score = (
    entity_grid_score * weights["entity_grid"] +
    semantic_score * weights["semantic_similarity"] +
    transition_score * weights["transitions"] +
    paragraph_unity * weights["paragraph_unity"]
)

print(f"Component Scores:")
print(f"  Entity Grid: {entity_grid_score:.2f}")
print(f"  Semantic Similarity: {semantic_score:.2f}")
print(f"  Transitions: {transition_score:.2f}")
print(f"  Paragraph Unity: {paragraph_unity:.2f}")
print(f"\nFinal Coherence Score: {coherence_score:.2f}/100")


## 4. Argument Miner Process

The Argument Miner uses Toulmin's Model of Argumentation:
- **Thesis** - Main argument statement
- **Claims** - Supporting assertions
- **Evidence/Grounds** - Supporting data/examples
- **Warrants** - Reasoning connecting evidence to claims
- **Rebuttals** - Counterarguments

### Step-by-Step Process:


In [ ]:
print("Step 1: Identifying Thesis Statement")
print("="*80)

# Thesis indicators
claim_indicators = [
    "i believe", "i think", "i argue", "thesis", "main point",
    "should", "must", "ultimately", "requires", "presents"
]

paragraphs = [p.strip() for p in sample_essay.split('\n\n') if p.strip()]
first_paragraph = paragraphs[0] if paragraphs else sample_essay
first_sentences = [sent.text.strip() for sent in nlp(first_paragraph).sents]

thesis = None
for sentence in first_sentences:
    sentence_lower = sentence.lower()
    for indicator in claim_indicators:
        if indicator in sentence_lower:
            thesis = {
                "sentence": sentence,
                "confidence": "high"
            }
            print(f"✅ Thesis found: {sentence[:80]}...")
            break
    if thesis:
        break

if not thesis and first_sentences:
    # Fallback: use first substantial sentence
    thesis = {
        "sentence": first_sentences[0],
        "confidence": "medium"
    }
    print(f"⚠️  Thesis (fallback): {first_sentences[0][:80]}...")


In [ ]:
# Step 2: Extract Claims
print("\nStep 2: Extracting Claims")
print("="*80)

claims = []
for i, sentence in enumerate(sentences_list):
    sentence_lower = sentence.lower()
    for indicator in claim_indicators:
        if indicator in sentence_lower:
            claims.append({
                "sentence_index": i,
                "sentence": sentence,
                "indicator": indicator
            })
            print(f"  Claim {len(claims)}: {sentence[:70]}...")
            break

# Also check for topic sentences (first sentence of paragraphs)
paragraphs = [p.strip() for p in sample_essay.split('\n\n') if p.strip()]
for para_idx, paragraph in enumerate(paragraphs):
    para_sentences = [sent.text.strip() for sent in nlp(paragraph).sents]
    if para_sentences:
        first_sent = para_sentences[0]
        # Check if already found
        if not any(c["sentence"] == first_sent for c in claims):
            if len(first_sent.split()) >= 10:
                claims.append({
                    "sentence_index": len(claims),
                    "sentence": first_sent,
                    "indicator": "topic_sentence"
                })
                print(f"  Claim {len(claims)} (topic sentence): {first_sent[:70]}...")

print(f"\nTotal Claims Found: {len(claims)}")


In [ ]:
# Step 3: Extract Evidence/Grounds
print("\nStep 3: Extracting Evidence/Grounds")
print("="*80)

evidence_indicators = [
    "for example", "for instance", "according to",
    "research shows", "studies indicate", "evidence suggests",
    "data shows", "demonstrates", "illustrates"
]

grounds = []
for i, sentence in enumerate(sentences_list):
    sentence_lower = sentence.lower()
    for indicator in evidence_indicators:
        if indicator in sentence_lower:
            grounds.append({
                "sentence_index": i,
                "sentence": sentence,
                "indicator": indicator
            })
            print(f"  Evidence {len(grounds)}: {sentence[:70]}...")
            break

# Also check for concrete examples (sentences with specific details)
for i, sentence in enumerate(sentences_list):
    if any(g["sentence"] == sentence for g in grounds):
        continue
    
    sent_doc = nlp(sentence)
    # Check for specific details (numbers, proper nouns)
    has_details = any(token.tag_ in ["CD", "NNP", "NNPS"] for token in sent_doc)
    # Check for concrete nouns
    concrete_nouns = ["cost", "system", "energy", "technology", "resource"]
    has_concrete = any(token.text.lower() in concrete_nouns for token in sent_doc)
    
    if has_details and has_concrete:
        grounds.append({
            "sentence_index": i,
            "sentence": sentence,
            "indicator": "concrete_instance"
        })
        print(f"  Evidence {len(grounds)} (concrete): {sentence[:70]}...")

print(f"\nTotal Evidence Found: {len(grounds)}")


In [ ]:
# Step 4: Extract Warrants (Reasoning)
print("\nStep 4: Extracting Warrants (Reasoning)")
print("="*80)

warrant_indicators = [
    "because", "since", "due to", "therefore", "thus",
    "consequently", "this means", "suggests that",
    "causes", "leads to", "results in"
]

warrants = []
for i, sentence in enumerate(sentences_list):
    sentence_lower = sentence.lower()
    for indicator in warrant_indicators:
        if indicator in sentence_lower:
            warrants.append({
                "sentence_index": i,
                "sentence": sentence,
                "indicator": indicator
            })
            print(f"  Warrant {len(warrants)}: {sentence[:70]}...")
            break

print(f"\nTotal Warrants Found: {len(warrants)}")


In [ ]:
# Step 5: Extract Rebuttals
print("\nStep 5: Extracting Rebuttals")
print("="*80)

rebuttal_indicators = [
    "however", "although", "even though", "despite",
    "nevertheless", "on the other hand", "in contrast",
    "some may argue", "critics claim", "yet", "but"
]

rebuttals = []
for i, sentence in enumerate(sentences_list):
    sentence_lower = sentence.lower()
    for indicator in rebuttal_indicators:
        if indicator in sentence_lower:
            rebuttals.append({
                "sentence_index": i,
                "sentence": sentence,
                "indicator": indicator
            })
            print(f"  Rebuttal {len(rebuttals)}: {sentence[:70]}...")
            break

print(f"\nTotal Rebuttals Found: {len(rebuttals)}")


In [ ]:
# Step 6: Calculate Argument Scores
print("\nStep 6: Calculating Argument Scores")
print("="*80)

# Claim score
claim_score = 50.0
if claims:
    claim_score += 20.0
if len(claims) >= 2:
    claim_score += 15.0
if thesis:
    claim_score += 15.0
claim_score = min(100.0, claim_score)

# Evidence score
evidence_score = 40.0 if grounds else 20.0
if len(grounds) >= 2:
    evidence_score += 20.0
if len(grounds) >= 3:
    evidence_score += 15.0
if claims:
    ratio = len(grounds) / len(claims)
    if ratio >= 1.5:
        evidence_score += 25.0
    elif ratio >= 1.0:
        evidence_score += 15.0
evidence_score = min(100.0, evidence_score)

# Warrant score
warrant_score = 50.0 if warrants else 40.0
if len(warrants) >= 2:
    warrant_score += 20.0
if claims and len(warrants) >= len(claims) * 0.5:
    warrant_score += 30.0
warrant_score = min(100.0, warrant_score)

# Rebuttal score
rebuttal_score = 70.0 if rebuttals else 60.0
if len(rebuttals) >= 2:
    rebuttal_score += 20.0
rebuttal_score = min(100.0, rebuttal_score)

# Overall argument score
weights = {
    "claim": 0.35,
    "evidence": 0.35,
    "warrant": 0.20,
    "rebuttal": 0.10
}

argument_score = (
    claim_score * weights["claim"] +
    evidence_score * weights["evidence"] +
    warrant_score * weights["warrant"] +
    rebuttal_score * weights["rebuttal"]
)

print(f"Component Scores:")
print(f"  Claims: {claim_score:.2f}/100 ({len(claims)} found)")
print(f"  Evidence: {evidence_score:.2f}/100 ({len(grounds)} found)")
print(f"  Warrants: {warrant_score:.2f}/100 ({len(warrants)} found)")
print(f"  Rebuttals: {rebuttal_score:.2f}/100 ({len(rebuttals)} found)")
print(f"\nFinal Argument Score: {argument_score:.2f}/100")


## 5. Knowledge Graph Builder Process

The Knowledge Graph Builder:
1. **Extracts Concepts** - Key nouns, noun phrases, and named entities
2. **Identifies Relationships** - Co-occurrence and semantic relationships
3. **Builds Network Graph** - Creates a NetworkX graph structure
4. **Calculates Metrics** - Connectivity, density, clustering

### Step-by-Step Process:


In [ ]:
import networkx as nx
from collections import Counter

print("Step 1: Extracting Key Concepts")
print("="*80)

# Extract noun phrases
noun_phrases = []
for chunk in doc.noun_chunks:
    if len(chunk.text.split()) <= 3 and len(chunk.text) > 4:
        noun_phrases.append(chunk.text.lower())

# Extract named entities
entities = []
for ent in doc.ents:
    if ent.label_ in ["PERSON", "ORG", "GPE", "EVENT", "PRODUCT"]:
        entities.append(ent.text.lower())

# Extract important nouns
important_nouns = []
for token in doc:
    if (token.pos_ in ["NOUN", "PROPN"] and 
        not token.is_stop and 
        not token.is_punct and
        len(token.text) > 3):
        important_nouns.append(token.lemma_.lower())

# Combine and count
all_concepts = noun_phrases + entities + important_nouns
concept_counts = Counter(all_concepts)

# Top concepts
top_concepts = concept_counts.most_common(15)
concepts = []
for concept_text, frequency in top_concepts:
    importance = frequency * len(concept_text.split())
    concepts.append({
        "text": concept_text,
        "frequency": frequency,
        "importance": importance
    })

print(f"Top 10 Concepts:")
for i, concept in enumerate(concepts[:10], 1):
    print(f"  {i}. {concept['text']} (frequency: {concept['frequency']}, importance: {concept['importance']:.1f})")


In [ ]:
# Step 2: Extract Relationships
print("\nStep 2: Extracting Relationships")
print("="*80)

concept_texts = {c["text"].lower() for c in concepts}
sentences_list = [sent.text for sent in doc.sents]

relationships = []
for i, sentence in enumerate(sentences_list):
    sentence_lower = sentence.lower()
    sentence_concepts = [c for c in concepts if c["text"] in sentence_lower]
    
    # Create relationships between concepts in same sentence
    for j, concept1 in enumerate(sentence_concepts):
        for concept2 in sentence_concepts[j+1:]:
            # Check if relationship exists
            existing = next(
                (r for r in relationships 
                 if ((r["source"] == concept1["text"] and r["target"] == concept2["text"]) or
                     (r["source"] == concept2["text"] and r["target"] == concept1["text"]))),
                None
            )
            
            if existing:
                existing["weight"] += 1
            else:
                # Determine relationship type
                rel_type = "related"
                if any(word in sentence_lower for word in ["causes", "leads to", "results in"]):
                    rel_type = "causes"
                elif any(word in sentence_lower for word in ["is", "are", "means"]):
                    rel_type = "defines"
                
                relationships.append({
                    "source": concept1["text"],
                    "target": concept2["text"],
                    "type": rel_type,
                    "weight": 1
                })

relationships.sort(key=lambda x: x["weight"], reverse=True)

print(f"Top 10 Relationships:")
for i, rel in enumerate(relationships[:10], 1):
    print(f"  {i}. {rel['source']} --[{rel['type']}]--> {rel['target']} (weight: {rel['weight']})")


In [ ]:
# Step 3: Build NetworkX Graph
print("\nStep 3: Building NetworkX Graph")
print("="*80)

G = nx.Graph()

# Add nodes
for concept in concepts:
    G.add_node(concept["text"], 
              frequency=concept["frequency"],
              importance=concept.get("importance", 0))

# Add edges
for rel in relationships:
    if G.has_node(rel["source"]) and G.has_node(rel["target"]):
        G.add_edge(rel["source"], rel["target"],
                  weight=rel["weight"],
                  type=rel["type"])

print(f"Graph Statistics:")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Density: {nx.density(G):.3f}")
print(f"  Connected Components: {len(list(nx.connected_components(G)))}")
print(f"  Average Clustering: {nx.average_clustering(G):.3f}")
print(f"  Is Connected: {nx.is_connected(G)}")


In [ ]:
# Step 4: Calculate Knowledge Graph Scores
print("\nStep 4: Calculating Knowledge Graph Scores")
print("="*80)

# Connectivity score
if G.number_of_nodes() > 1:
    density = nx.density(G)
    connectivity_bonus = 20.0 if nx.is_connected(G) else 0.0
    connectivity_score = (density * 80) + connectivity_bonus
else:
    connectivity_score = 0.0

# Depth score
concept_score = min(50, len(concepts) * 3)
if concepts:
    rels_per_concept = len(relationships) / len(concepts)
    relationship_score = min(50, rels_per_concept * 10)
else:
    relationship_score = 0.0

depth_score = concept_score + relationship_score

# Overall score
kg_score = (connectivity_score + depth_score) / 2

print(f"Component Scores:")
print(f"  Connectivity: {connectivity_score:.2f}/100")
print(f"  Depth: {depth_score:.2f}/100")
print(f"\nFinal Knowledge Graph Score: {kg_score:.2f}/100")


## Summary: Complete Analysis Pipeline

Now let's see how all modules work together:


In [ ]:
print("="*80)
print("COMPLETE ANALYSIS RESULTS")
print("="*80)

# Compile all scores
scores = {
    "grammar": grammar_score,
    "readability": readability_score,
    "coherence": coherence_score,
    "argument_strength": argument_score,
    "knowledge_graph": kg_score
}

# Calculate overall score
weights = {
    "grammar": 0.20,
    "readability": 0.20,
    "coherence": 0.25,
    "argument_strength": 0.25,
    "knowledge_graph": 0.10
}

overall_score = sum(scores[dim] * weights[dim] for dim in scores.keys())

print("\nDimension Scores:")
for dimension, score in scores.items():
    print(f"  {dimension.replace('_', ' ').title()}: {score:.2f}/100")

print(f"\nOverall Score: {overall_score:.2f}/100")

print("\n" + "="*80)
print("ANALYSIS BREAKDOWN")
print("="*80)
print(f"\nGrammar:")
print(f"  - Errors detected: {len(grammar_errors)}")
print(f"  - Syntax complexity: {complexity_score:.2f}")

print(f"\nReadability:")
print(f"  - Flesch Reading Ease: {flesch_ease:.2f}")
print(f"  - Grade Level: {flesch_grade:.2f}")
print(f"  - Lexical Diversity: {lexical_diversity:.2%}")

print(f"\nCoherence:")
print(f"  - Entity continuity: {entity_grid_score:.2f}")
print(f"  - Semantic similarity: {semantic_score:.2f}")
print(f"  - Transitions: {transition_score:.2f}")

print(f"\nArgumentation:")
print(f"  - Thesis: {'Found' if thesis else 'Not found'}")
print(f"  - Claims: {len(claims)}")
print(f"  - Evidence: {len(grounds)}")
print(f"  - Warrants: {len(warrants)}")
print(f"  - Rebuttals: {len(rebuttals)}")

print(f"\nKnowledge Graph:")
print(f"  - Concepts: {len(concepts)}")
print(f"  - Relationships: {len(relationships)}")
print(f"  - Graph density: {nx.density(G):.3f}")


## Key Takeaways

### How Each Module Works:

1. **Grammar Analyzer**
   - Uses LanguageTool for rule-based grammar checking
   - Uses spaCy for syntactic analysis (dependency parsing)
   - Calculates error density to determine score

2. **Readability Analyzer**
   - Uses textstat library for standard readability formulas
   - Calculates lexical diversity (vocabulary variety)
   - Normalizes scores for academic writing appropriateness

3. **Coherence Analyzer**
   - Entity-grid model tracks noun continuity across sentences
   - SentenceTransformer provides semantic similarity
   - Transition words indicate logical flow
   - Paragraph unity measures topic sentence support

4. **Argument Miner**
   - Pattern matching for Toulmin model components
   - Optional transformer-based classification (if available)
   - Scores based on presence and quality of argument components

5. **Knowledge Graph Builder**
   - Extracts concepts using NLP (nouns, entities, noun phrases)
   - Builds relationships from co-occurrence
   - NetworkX graph structure for analysis
   - Metrics: connectivity, density, clustering

### Processing Flow:

```
Input Text
    ↓
┌─────────────────────────────────────┐
│  Essay Analysis Service             │
│  (Orchestrates all modules)        │
└─────────────────────────────────────┘
    ↓
┌──────────┬──────────┬──────────┬──────────┬──────────┐
│ Grammar  │Readability│Coherence│ Argument │Knowledge │
│ Analyzer │ Analyzer  │ Analyzer │  Miner   │  Graph   │
└──────────┴──────────┴──────────┴──────────┴──────────┘
    ↓
┌─────────────────────────────────────┐
│  Aggregate Scores & Recommendations│
└─────────────────────────────────────┘
    ↓
Output: Complete Analysis Report
```

### Technologies Used:

- **spaCy**: NLP processing, dependency parsing, POS tagging
- **LanguageTool**: Grammar checking
- **SentenceTransformer**: Semantic embeddings
- **NetworkX**: Graph analysis
- **textstat**: Readability metrics
- **NLTK**: Text processing utilities
